<a href="https://colab.research.google.com/github/Satishpk15/Machine-learning/blob/main/Zepto_%26_Blinkit_workshop_14th_June_2026_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ How Analysts at Zepto & Blinkit Make Decisions in Minutes
### Hands-on Data Visualization — Matplotlib & Seaborn

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global chart defaults — run this once, applies to everything below (Run Commands Parameters is a dictionary of Matplotlib Configuration settings)
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

# Shared palette and named colors — we'll use these throughout
PALETTE = sns.color_palette("Set2")
ORANGE  = "#E67E22"
BLUE    = "#2980B9"
RED     = "#E74C3C"
GREEN   = "#27AE60"

print("✅ Libraries loaded and defaults set")

✅ Libraries loaded and defaults set


In [ ]:
# ─── Dataset Generation ───────────────────────────────────────────────────────
# Generates zepto_orders.csv — run once per Colab session.

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(42)
N = 3500

cities = ["Bengaluru","Mumbai","Delhi","Hyderabad","Pune","Chennai"]
city_w = [0.25, 0.20, 0.20, 0.15, 0.10, 0.10]

dark_stores = {
    "Bengaluru": ["Koramangala DS","Indiranagar DS","HSR Layout DS"],
    "Mumbai":    ["Andheri DS","Bandra DS","Powai DS"],
    "Delhi":     ["CP Dark Store","Lajpat Nagar DS","Dwarka DS"],
    "Hyderabad": ["Banjara Hills DS","Gachibowli DS"],
    "Pune":      ["Kothrud DS","Wakad DS"],
    "Chennai":   ["T Nagar DS","Velachery DS"],
}

categories = ["Fruits & Vegetables","Dairy & Breakfast","Snacks & Munchies",
              "Beverages","Personal Care","Household","Staples & Grains","Frozen Veg"]
cat_w = [0.18, 0.22, 0.14, 0.12, 0.10, 0.08, 0.11, 0.05]

products = {
    "Fruits & Vegetables": ["Tomatoes 500g","Onions 1kg","Potatoes 1kg","Spinach Bunch",
                            "Bananas 6pcs","Apples 4pcs","Capsicum 3pcs"],
    "Dairy & Breakfast":   ["Amul Milk 1L","Curd 400g","Paneer 200g","Amul Butter 100g",
                            "Britannia Cheese Slices","Quaker Oats 500g","Bread Loaf"],
    "Snacks & Munchies":   ["Lays Classic 26g","Kurkure Masala","Parle-G 250g",
                            "Haldiram's Namkeen","Popcorn Butter","Digestive Biscuits"],
    "Beverages":           ["Real Mango Juice","Coconut Water","Limca 750ml",
                            "Paper Boat Aam Panna","Horlicks 200g","Minute Maid"],
    "Personal Care":       ["Dove Shampoo 180ml","Himalaya Face Wash","Colgate Toothpaste",
                            "Dettol Soap","Nivea Body Lotion","Sunscreen SPF50"],
    "Household":           ["Vim Dish Wash 500g","Harpic Floor Cleaner","Tissue Paper Pack",
                            "Garbage Bags 30pcs","Lizol Disinfectant 500ml"],
    "Staples & Grains":    ["Basmati Rice 1kg","Aashirvaad Atta 1kg","Toor Dal 500g",
                            "Sugar 1kg","Saffola Gold Oil 1L","Iodized Salt 1kg"],
    "Frozen Veg":          ["Veg Momos 400g","Frozen Peas 500g","Corn Kernels 500g",
                            "Mixed Vegetables 500g","Edamame 200g"],
}

price_range = {
    "Fruits & Vegetables": (25,120), "Dairy & Breakfast":(30,200),
    "Snacks & Munchies":(15,80),     "Beverages":(20,130),
    "Personal Care":(60,380),        "Household":(50,260),
    "Staples & Grains":(40,220),     "Frozen Veg":(80,250),
}

dt_params = {
    "Bengaluru":(22,7), "Mumbai":(20,6), "Delhi":(18,5),
    "Hyderabad":(16,4), "Pune":(14,4),   "Chennai":(17,5),
}

payment_modes = ["UPI","Card","Cash","Wallet"]
pay_w         = [0.60, 0.22, 0.10, 0.08]

hour_probs = np.array([
    0.005,0.003,0.002,0.001,0.002,0.005,
    0.012,0.032,0.055,0.045,0.028,0.022,
    0.038,0.055,0.040,0.030,0.035,0.048,
    0.065,0.080,0.090,0.075,0.055,0.030,
])
hour_probs /= hour_probs.sum()

hour_probs_wknd = hour_probs.copy()
hour_probs_wknd[7:10] *= 1.5
hour_probs_wknd[18:22] *= 1.3
hour_probs_wknd /= hour_probs_wknd.sum()

slot_dt_mult = {
    "Morning (6-11am)":0.90, "Afternoon (11am-4pm)":0.95,
    "Evening (4-9pm)":1.18,  "Night (9pm+)":1.05,
}

def get_slot(h):
    if 6<=h<11:  return "Morning (6-11am)"
    if 11<=h<16: return "Afternoon (11am-4pm)"
    if 16<=h<21: return "Evening (4-9pm)"
    return "Night (9pm+)"

rows = []
base = datetime(2024, 11, 1)
for i in range(N):
    city     = rng.choice(cities, p=city_w)
    store    = rng.choice(dark_stores[city])
    category = rng.choice(categories, p=cat_w)
    product  = rng.choice(products[category])
    lo,hi    = price_range[category]
    mrp      = float(rng.integers(lo, hi+1))
    disc_pct = float(rng.choice([0,5,10,15,20,25],p=[0.35,0.15,0.20,0.15,0.10,0.05]))
    sell_px  = round(mrp*(1-disc_pct/100), 2)
    qty      = int(rng.choice([1,2,3], p=[0.65,0.25,0.10]))

    order_day = int(rng.integers(0, 30))
    tmp_date  = base + timedelta(days=order_day)
    weekday   = tmp_date.strftime('%a')
    probs     = hour_probs_wknd if weekday in ('Sat','Sun') else hour_probs
    hour      = int(rng.choice(24, p=probs))
    slot      = get_slot(hour)
    order_time = tmp_date + timedelta(hours=hour, minutes=int(rng.integers(0,60)))

    status = rng.choice(["Delivered","Cancelled"], p=[0.95, 0.05])
    mu,sd  = dt_params[city]
    dt     = int(np.clip(rng.normal(mu*slot_dt_mult[slot], sd), 7, 50)) if status=="Delivered" else np.nan
    rating = float(np.round(np.clip(rng.normal(4.0,0.7),1.0,5.0),1))
    if status=="Cancelled" or rng.random()<0.12:
        rating = np.nan

    rows.append({
        "order_id":f"ZBL{200000+i}", "order_time":order_time,
        "hour":hour, "day_of_week":weekday, "slot":slot,
        "city":city, "dark_store":store, "category":category, "product":product,
        "mrp":mrp, "selling_price":sell_px, "discount_pct":disc_pct,
        "quantity":qty, "order_total":round(sell_px*qty,2),
        "delivery_time_min":dt, "promised_min":10,
        "payment_mode":rng.choice(payment_modes, p=pay_w),
        "rating":rating, "status":status,
    })

df = pd.DataFrame(rows)
df.to_csv("zepto_orders.csv", index=False)
print(f"✅ zepto_orders.csv — {len(df):,} rows, {df.shape[1]} columns")
print(f"   Delivered: {(df.status=='Delivered').sum():,}  |  Cancelled: {(df.status=='Cancelled').sum():,}")
print(f"   Missing delivery_time: {df.delivery_time_min.isna().sum()}  |  Missing rating: {df.rating.isna().sum()}")


✅ zepto_orders.csv — 3,500 rows, 19 columns
   Delivered: 3,301  |  Cancelled: 199
   Missing delivery_time: 199  |  Missing rating: 577


**Key columns:**
- `hour`, `day_of_week`, `slot` — *when* things happen
- `city`, `dark_store` — *where*
- `category`, `product`, `order_total` — *what* and *how much*
- `delivery_time_min`, `promised_min` — *performance* (our main story)
- `rating` — *customer feedback*

**Our premise:** Zepto/Blinkit promise **10-minute delivery**. All the charts today either prove or disprove that — and explain why.


In [ ]:
# Load and get a quick look
df = pd.read_csv("zepto_orders.csv", parse_dates=["order_time"])

# We'll use 'delivered' for most charts — cancelled orders have no delivery time
delivered = df[df["status"] == "Delivered"].copy()

print("Shape:", df.shape)
df.head()

Shape: (3500, 19)


,order_id,order_time,hour,day_of_week,slot,city,dark_store,category,product,mrp,selling_price,discount_pct,quantity,order_total,delivery_time_min,promised_min,payment_mode,rating,status
0,ZBL200000,2024-11-21 20:43:00,20,Thu,Evening (4-9pm),Hyderabad,Gachibowli DS,Staples & Grains,Toor Dal 500g,55.0,55.0,0.0,3,165.0,15.0,10,Card,4.6,Delivered
1,ZBL200001,2024-11-26 19:49:00,19,Tue,Evening (4-9pm),Pune,Wakad DS,Snacks & Munchies,Parle-G 250g,29.0,26.1,10.0,1,26.1,13.0,10,Card,4.9,Delivered
2,ZBL200002,2024-11-05 23:24:00,23,Tue,Night (9pm+),Bengaluru,Indiranagar DS,Fruits & Vegetables,Spinach Bunch,77.0,69.3,10.0,2,138.6,27.0,10,UPI,4.8,Delivered
3,ZBL200003,2024-11-03 21:18:00,21,Sun,Night (9pm+),Bengaluru,Koramangala DS,Personal Care,Sunscreen SPF50,200.0,170.0,15.0,2,340.0,23.0,10,UPI,4.2,Delivered
4,ZBL200004,2024-11-20 16:30:00,16,Wed,Evening (4-9pm),Bengaluru,HSR Layout DS,Household,Vim Dish Wash 500g,214.0,181.9,15.0,2,363.8,36.0,10,UPI,3.4,Delivered
